# Worksheet 2, Part 3 — Summary plot
### HE 1002-0755 vs the six standard stars

In [ ]:
from pylab import *
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from astropy.io import fits, ascii
from astropy.table import Table
import numpy as np, os

DATA = "../../../data"
os.makedirs("plots", exist_ok=True)

# set this once the merged rest-frame spectrum exists (SMH .txt export or a _RV_corr.fits)
OUR_STAR = f"{DATA}/he1002-0755_coadd_rest.fits"   # rest-frame coadd of all 10 epochs
OUR_NOTE  = "coadd of 10 epochs"
OUR_LABEL = "HE 1002-0755"

In [ ]:
plt.rcParams['font.size']           = 14
plt.rcParams['lines.linewidth']     = 1.0
plt.rcParams["xtick.major.size"]    = 8.0
plt.rcParams["ytick.major.size"]    = 8.0
plt.rcParams["xtick.minor.size"]    = 5.0
plt.rcParams["ytick.minor.size"]    = 5.0
plt.rcParams["xtick.labelsize"]     = 13
plt.rcParams["ytick.labelsize"]     = 13
plt.rcParams["axes.labelsize"]      = 15
plt.rcParams['axes.linewidth']      = 1.6
plt.rcParams["xtick.direction"]     = "in"
plt.rcParams["ytick.direction"]     = "in"
plt.rcParams["xtick.top"]           = True
plt.rcParams["ytick.right"]         = True
plt.rcParams["xtick.minor.visible"] = True
plt.rcParams["ytick.minor.visible"] = True

## The standard stars

In [ ]:
# Wong colour-blind-safe palette; standards ordered by literature temperature
# so the legend itself reads as a temperature sequence.
STANDARDS = [
    # file,              label,           T_lit, [Fe/H], colour
    ("HD122563.fits",    "HD 122563",      4600,  -2.7,  "#0072B2"),
    ("he1523-0901.fits", "HE 1523-0901",   4630,  -3.0,  "#D55E00"),
    ("cs22892-52.fits",  "CS 22892-052",   4800,  -3.1,  "#009E73"),
    ("cd-38_245.fits",   "CD -38 245",     4800,  -4.1,  "#E69F00"),
    ("HD140283.fits",    "HD 140283",      5650,  -2.5,  "#CC79A7"),
    ("G64-12.fits",      "G64-12",         6450,  -3.2,  "#56B4E9"),
]

def load_fits1d(path):
    h = fits.open(path)[0]; hd = h.header
    n  = hd["NAXIS1"]; cd = hd.get("CD1_1", hd.get("CDELT1"))
    w  = hd["CRVAL1"] + cd*(np.arange(n) + 1 - hd.get("CRPIX1", 1))
    return w, np.asarray(h.data, float)

spec = {}
for fn, lab, T, feh, col in STANDARDS:
    spec[lab] = load_fits1d(f"{DATA}/{fn}")
    w, f = spec[lab]
    print(f"  {lab:<13s} T={T:<5d} [Fe/H]={feh:<5.1f}  {w[0]:.0f}-{w[-1]:.0f} A")

ours = None
if OUR_STAR:
    if OUR_STAR.endswith(".txt"):
        t = ascii.read(OUR_STAR); ours = (np.asarray(t["col1"], float), np.asarray(t["col2"], float))
    else:
        t = Table.read(OUR_STAR)
        fc = "norm_flux" if "norm_flux" in t.colnames else "flux"
        ours = (np.asarray(t["wavelength"], float), np.asarray(t[fc], float))
    print(f"\n  our star: {OUR_LABEL}  {ours[0][0]:.0f}-{ours[0][-1]:.0f} A")

## The summary plot

In [ ]:
plt.rcParams.update({
    "font.family": "serif", "font.size": 9,
    "axes.labelsize": 10, "axes.linewidth": 0.9,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.top": True, "ytick.right": True,
    "xtick.minor.visible": True, "ytick.minor.visible": True,
    "legend.frameon": False,
})

PANELS = [
    (r"H$\beta$ 4861",   4859.0, 4864.0, (0.0, 1.30), 1),
    (r"H$\alpha$ 6563",  6559.5, 6565.5, (0.0, 1.30), 1),
    ("Mg b 5130-5190",   5130.0, 5190.0, (0.0, 1.30), 10),
    ("Pb I 4057",        4056.0, 4058.0, (0.55, 1.20), 0.5),
    ("CH G band 4310-4325", 4310.0, 4325.0, (0.0, 1.45), 3),
    ("Eu II 4126-4133",  4126.0, 4133.0, (0.0, 1.30), 2),
    ("Th II 4019.1",     4018.0, 4020.0, (0.55, 1.20), 0.5),
    ("Ba II 4554",       4553.0, 4555.0, (0.0, 1.30), 0.5),
]

fig, axes = plt.subplots(4, 2, figsize=(9.5, 10.5))
for ax, (name, lo, hi, ylim, dtick) in zip(axes.ravel(), PANELS):
    for fn, lab, T, feh, col in STANDARDS:
        w, f = spec[lab]
        m = (w > lo - 1) & (w < hi + 1)
        if m.sum() > 3:
            ax.plot(w[m], f[m], color=col, lw=0.85, alpha=0.9,
                    label=f"{lab}  {T} K, [Fe/H] {feh:+.1f}")
    if ours is not None:
        w, f = ours
        m = (w > lo - 1) & (w < hi + 1)
        if m.sum() > 3:
            ax.plot(w[m], f[m], color="black", lw=1.8, label=OUR_LABEL, zorder=10)
    ax.set_xlim(lo, hi); ax.set_ylim(*ylim)
    ax.xaxis.set_major_locator(plt.MultipleLocator(dtick))
    ax.text(0.025, 0.06, name, transform=ax.transAxes, fontsize=10,
            weight="bold", va="bottom")

for ax in axes[-1]:
    ax.set_xlabel(r"Rest wavelength, $\lambda$ ($\AA$)")
for ax in axes[:, 0]:
    ax.set_ylabel(r"Normalised flux")

h, l = axes[0, 0].get_legend_handles_labels()
order = list(range(1, len(l))) + [0] if l[0] == OUR_LABEL else list(range(len(l)))
h = [h[i] for i in order]; l = [l[i] for i in order]
fig.legend(h, l, loc="upper center", ncol=4, fontsize=8.5,
           bbox_to_anchor=(0.5, 0.995), handlelength=1.8, columnspacing=1.4)
fig.suptitle(f"{OUR_LABEL} (rest-frame coadd of 10 MIKE exposures) vs six standards",
             y=1.035, fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.955])
for ext in ("png", "pdf"):
    fig.savefig(f"plots/summary_plot_HE1002-0755.{ext}", dpi=200, bbox_inches="tight")
plt.show()
print("  saved plots/summary_plot_HE1002-0755.{png,pdf}")

## Reading the plot (worksheet procedure)

## The Balmer pair on its own

In [ ]:
# Worksheet 2 Part 3 section 1.1: the two Balmer lines on their own, larger,
# because the temperature estimate is read off these wings.
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2))
for ax, (name, lo, hi) in zip(axes, [(r"H$\beta$ 4861", 4857.0, 4866.0),
                                     (r"H$\alpha$ 6563", 6557.0, 6569.0)]):
    for fn, lab, T, feh, col in STANDARDS:
        w, f = spec[lab]
        m = (w > lo - 1) & (w < hi + 1)
        ax.plot(w[m], f[m], color=col, lw=1.0, alpha=0.9, label=f"{lab}  {T} K")
    if ours is not None:
        w, f = ours
        m = (w > lo - 1) & (w < hi + 1)
        ax.plot(w[m], f[m], color="black", lw=2.0, label=OUR_LABEL, zorder=10)
    ax.set_xlim(lo, hi); ax.set_ylim(0, 1.25)
    ax.set_xlabel(r"Rest wavelength, $\lambda$ ($\AA$)")
    ax.text(0.03, 0.06, name, transform=ax.transAxes, fontsize=11, weight="bold")
axes[0].set_ylabel("Normalised flux")
axes[1].legend(fontsize=8, loc="lower right")
fig.suptitle("Balmer profiles: the temperature diagnostic\n(HE 1002-0755 = rest-frame coadd of 10 MIKE exposures)", y=1.0, fontsize=12)
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(f"plots/balmer_profiles_HE1002-0755.{ext}", dpi=200, bbox_inches="tight")
plt.show()
print("  saved plots/balmer_profiles_HE1002-0755.{png,pdf}")